# Modern Transformer Architecture - Homework

In this assignment, you'll implement key innovations in modern Transformers:
- **RMSNorm**: Simpler and faster normalization
- **RoPE**: Rotary positional embeddings for better length extrapolation
- **Simplified MHLA**: Efficient attention with reduced KV cache

**What you'll learn:**
- Why modern LLMs use these techniques
- How they improve efficiency and performance
- Trade-offs between different approaches

**Submission:** Submit this completed notebook and completed python files.

**Grading:**
- Part 1 (RMSNorm): 20 points
- Part 2 (RoPE): 30 points
- Part 3 (MHLA): 35 points
- Part 4 (Integration + Questions): 15 points
Total: 100 points

## Setup

See `runner.py`.

## Part 1: RMSNorm (20 points)

See `RMSNorm.py`.

See `runner.py`.

**Question 1.1 (5 points - Written):** Why doesn't RMSNorm need to compute the mean? In what way is "controlling scale" sufficient for gradient stability?

**Why RMSNorm doesn't need to compute the mean**: RMSNorm is a variation of LayerNorm that is less computationally intensive because it utilizes the fact that only scaling is necessary for effective normalization for gradient stability. Normalizing the mean turns out to be extraneous to the desired output, and so RMSNorm drops it.

**In what way "controlling scale" is sufficient for gradient stability**: This is sufficient for gradient stability because the rescaling of activations turns out to be sufficent for preventing vanishing or exploding gradients. As the model moves through its layers, the activations are rescaled relative to the root-mean-squared so large gradients self-correct. The model the learns the variable that it uses to control the scaling, meaning it can optimize for scaling without worrying about accumulating or vanishing gradients.

## Part 2: Rotary Positional Embeddings (30 points)

See `RoPE.py`.

See `runner.py`.

**Question 2.1 (5 points - Written):** Why does RoPE generalize better to longer sequences than learned absolute positional embeddings?

**Why RoPE generalizes better to longer sequences than learned absolute positional embeddings**: RoPE generalizes better to longer sequences because the way that it encodes position is relative positions between tokens. The absolute positions are not encoded but distances between tokens are, so the model can learn positional patterns regardless of where those patterns are. With absolute position embeddings, if a patern were even one unit over from the learned position, the model would not easily recognize it. With RoPE, that difference is immaterial. For example, it can detect that two words are often close to one another, like "neither" and "nor", regardless of where they appear in the sequence. This means that with longer sequences (those longer than seen in training), the model will still be able to evaluate relationships between words and their positional importance. With absolute positional embeddings, the model would not have information for anything beyond the positions it learned.

# Part 3: Simplified Multi-Head Latent Attention (35 points)

See `MHLA.py`.

See `runner.py`.

**Question 3.1 (5 points - Written):** The compression ratio is constant regardless of sequence
length. Why? What does this tell you about where the memory savings come from?

**Why the compression ratio is constant regardless of sequence length**: MHLA and the standard attention mechanism caches both have sizes that multiply the sequence length by a constant; the difference is what that constant is. When the ratio of their compression is calculated, the sequence length cancels out, leaving just the two different multipliers.

**What this tells about where memory savings come from**: This tells me that the memory savings come because the number of features cached per token is smaller. It allows the designer to control the amount of memory used per token instead of letting it blow up on larger models.

**Question 3.2 (5 points - Written):** MHLA compresses K and V significantly. What information
might be lost? In what scenarios might this hurt model quality?

**What information might be lost**: More specific, token level information could be lost because we limit the amount of information we can represent per token. Anything happening infrequently or with high precision could fail to be captured by this generalization.

**In what scenarios might this hurt model quality**: This could hurt quality when needing to track information with a high level of specifitity. An example of something MHLa could struggle on is learning about British history, where the specific generational suffixes attached to common names like Mary and Henry refer to very different people and times in history. 

## Part 4: Putting It All Together (15 points)

See `Transformer.py`.

See `runner.py`.

## Final Questions

### Final Questions

**Question 4.1 (3 points - Written):** Imagine you're deploying a chatbot that needs to handle
conversations of 50,000 tokens. Would you use standard attention or MHLA?
Why? What would be the memory savings?

**Which of standard attention or MHLA you would use**: I would use MHLA.

**Why**: The model would need to be able to maintain memory while it handles a long conversation, so the use of a model that can control for memory useage would be better.

(It would depend on the expected content of the conversations, but if they were general (i.e. not specifically a chatbot for law students or software developers), the conversations likely do not need to handle many extreme specifics.)

**What the memory savings would be**: The memory savings would be `the latent dimension / (2*the model size)`.

**Question 4.2 (3 points - Written):** What's the main trade-off when using a smaller latent dimension
$d_{\text{latent}}$ in MHLA? How would you choose this hyperparameter?

**What the main trade-off when using a smaller latent dimension in MHLA is**: The main tradeoff is the ability to retain token-level information vs controlling memory usage over time. Smaller dimension means the model can retain less specificity in attention. It also means that the model costs less memory as it uses attention, so it will not run out of space for further conversastion.

**How you would choose the latent dimension**: I would want to choose the maximum latent dimension that my model could run on while still maintaining speed and memory efficiency on the available hardware.

**Question 4.3 (4 points - Written):** We used Pre-LN (normalize before the block) instead of Post-LN (normalize after the residual). Why is Pre-LN better for deep networks? Hint: think about gradient flow.

**Why Pre-LN is better for deep networks**: With pre-LN, the identity term (x) stays throughout the flow. This means the gradient will always pass to earlier blocks in the network with out being scaled. After many blocks, the stacked scaling of many layers can lead to vanishing or exploding gradients. Pre-LN on deep networks lets the gradients stay stable by letting the identity term prevent too much stacking compromise gradient stability.